# Import Libraries

In [2]:
import os
import glob
import joblib
import numpy as np
import pandas as pd
import warnings

from sklearn.metrics import average_precision_score, confusion_matrix
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Find the fold files

In [3]:
# Find the uploaded Experiment 1 fold files

rf_files = sorted(
    glob.glob("/content/Random_Forest/fold*.joblib")
)

lgbm_files = sorted(
    glob.glob("/content/LightGBM/fold*.joblib")
)

print("Random Forest files:")
for f in rf_files:
    print(os.path.basename(f))

print("\nLightGBM files:")
for f in lgbm_files:
    print(os.path.basename(f))

if len(rf_files) != 5:
    raise FileNotFoundError(
        f"Expected 5 Random Forest fold files, found {len(rf_files)}"
    )

if len(lgbm_files) != 5:
    raise FileNotFoundError(
        f"Expected 5 LightGBM fold files, found {len(lgbm_files)}"
    )

Random Forest files:
fold1.joblib
fold2.joblib
fold3.joblib
fold4.joblib
fold5.joblib

LightGBM files:
fold1.joblib
fold2.joblib
fold3.joblib
fold4.joblib
fold5.joblib


In [4]:
print(rf_files)
print(lgbm_files)

['/content/Random_Forest/fold1.joblib', '/content/Random_Forest/fold2.joblib', '/content/Random_Forest/fold3.joblib', '/content/Random_Forest/fold4.joblib', '/content/Random_Forest/fold5.joblib']
['/content/LightGBM/fold1.joblib', '/content/LightGBM/fold2.joblib', '/content/LightGBM/fold3.joblib', '/content/LightGBM/fold4.joblib', '/content/LightGBM/fold5.joblib']


# Load Dataesets

In [5]:
ds1 = pd.read_csv( "/content/CKD_preprocessed.csv")
ds2 = pd.read_csv("/content/DiabeticCKD_engineered.csv")

print("Dataset 1:", ds1.shape)
print("Dataset 2:", ds2.shape)

Dataset 1: (380, 21)
Dataset 2: (4000, 34)


In [6]:
ds1["gender_group"] = (
    ds1["gender"]
    .map({
        0: "Female",
        1: "Male"
    })
)

ds2["gender_group"] = (
    ds2["Gender"]
    .map({
        0: "Female",
        1: "Male"
    })
)

def broad_age_group(age):
    if age < 40:
        return "<40"
    elif age < 50:
        return "40-49"
    else:
        return "50+"

ds1["age_group"] = ds1["age"].apply(broad_age_group)
ds2["age_group"] = ds2["Age"].apply(broad_age_group)


print("Dataset 1 gender:")
print(ds1["gender_group"].value_counts(dropna=False))

print("\nDataset 1 age:")
print(ds1["age_group"].value_counts())

print("\nDataset 2 gender:")
print(ds2["gender_group"].value_counts(dropna=False))

print("\nDataset 2 age:")
print(ds2["age_group"].value_counts())

Dataset 1 gender:
gender_group
Female    195
Male      185
Name: count, dtype: int64

Dataset 1 age:
age_group
<40      251
40-49     78
50+       51
Name: count, dtype: int64

Dataset 2 gender:
gender_group
Female    2550
Male      1450
Name: count, dtype: int64

Dataset 2 age:
age_group
50+      2184
40-49    1240
<40       576
Name: count, dtype: int64


# Inspect a fold file

In [7]:
rf_test = joblib.load(rf_files[0])
lgbm_test = joblib.load(lgbm_files[0])

print("Random Forest fold 1:")
print(type(rf_test))

if isinstance(rf_test, dict):
    print(rf_test.keys())

print("\nLightGBM fold 1:")
print(type(lgbm_test))

if isinstance(lgbm_test, dict):
    print(lgbm_test.keys())

Random Forest fold 1:
<class 'dict'>
dict_keys(['fold', 'best_pipeline', 'best_params', 'threshold', 'train_indices', 'test_indices', 'inner_best_score', 'y_true', 'y_prob', 'y_pred'])

LightGBM fold 1:
<class 'dict'>
dict_keys(['fold', 'best_pipeline', 'best_params', 'threshold', 'train_indices', 'test_indices', 'inner_best_score', 'y_true', 'y_prob', 'y_pred'])


# Extract the results

In [8]:
def extract_fold_result(path):

    data = joblib.load(path)

    if not isinstance(data, dict):
        raise TypeError(
            f"{path} does not contain a dictionary."
        )


    test_indices = data["test_indices"]
    y_true = data["y_true"]
    y_prob = data["y_prob"]
    threshold = data["threshold"]

    if test_indices is None:
        raise KeyError(
            f"No test indices found in {path}. "
            f"Available keys: {list(data.keys())}"
        )

    if y_true is None:
        raise KeyError(
            f"No true labels found in {path}. "
            f"Available keys: {list(data.keys())}"
        )

    if y_prob is None:
        raise KeyError(
            f"No predicted probabilities found in {path}. "
            f"Available keys: {list(data.keys())}"
        )

    if threshold is None:
        raise KeyError(
            f"No threshold found in {path}. "
            f"Available keys: {list(data.keys())}"
        )

    return (
        np.asarray(test_indices),
        np.asarray(y_true),
        np.asarray(y_prob),
        float(threshold)
    )

# CKD dataset predictions

In [9]:
ds1_predictions = []

for fold, path in enumerate(rf_files, start=1):

    test_indices, y_true, y_prob, threshold = (extract_fold_result(path))

    fold_df = pd.DataFrame({
        "original_index": test_indices,
        "fold": fold,
        "y_true": y_true,
        "y_prob": y_prob,
        "threshold": threshold
    })

    fold_df["y_pred"] = (fold_df["y_prob"] >= fold_df["threshold"]).astype(int)
    ds1_predictions.append(fold_df)

ds1_predictions = pd.concat(
    ds1_predictions,
    ignore_index=True
)

print(ds1_predictions.shape)

(380, 6)


In [10]:
ds1_predictions = ds1_predictions.merge(
    ds1[
        [
            "gender_group",
            "age_group"
        ]
    ].reset_index().rename(
        columns={
            "index": "original_index"
        }
    ),
    on="original_index",
    how="left"
)

print(ds1_predictions.head())

   original_index  fold  y_true    y_prob  threshold  y_pred gender_group  \
0              24     1       1  0.800280       0.49       1       Female   
1              26     1       0  0.625637       0.49       1       Female   
2              28     1       0  0.240416       0.49       0         Male   
3              31     1       0  0.041497       0.49       0       Female   
4              37     1       1  0.974334       0.49       1         Male   

  age_group  
0       <40  
1       <40  
2       <40  
3       <40  
4       <40  


# DiabeticCKD dataset predictions

In [11]:
ds2_predictions = []

for fold, path in enumerate(lgbm_files, start=1):

    test_indices, y_true, y_prob, threshold = (
        extract_fold_result(path)
    )

    fold_df = pd.DataFrame({
        "original_index": test_indices,
        "fold": fold,
        "y_true": y_true,
        "y_prob": y_prob,
        "threshold": threshold
    })

    fold_df["y_pred"] = (
        fold_df["y_prob"] >= fold_df["threshold"]
    ).astype(int)

    ds2_predictions.append(fold_df)


ds2_predictions = pd.concat(
    ds2_predictions,
    ignore_index=True
)

print(ds2_predictions.shape)

(4000, 6)


In [12]:
ds2_predictions = ds2_predictions.merge(
    ds2[
        [
            "gender_group",
            "age_group"
        ]
    ].reset_index().rename(
        columns={
            "index": "original_index"
        }
    ),
    on="original_index",
    how="left"
)

print(ds2_predictions.head())

   original_index  fold  y_true    y_prob  threshold  y_pred gender_group  \
0              10     1       0  0.009817       0.25       0       Female   
1              11     1       0  0.006535       0.25       0       Female   
2              12     1       0  0.005787       0.25       0       Female   
3              13     1       0  0.006681       0.25       0       Female   
4              14     1       0  0.031793       0.25       0       Female   

  age_group  
0       50+  
1       50+  
2       50+  
3       50+  
4       50+  


# Subgroup metric function

In [13]:
MIN_POSITIVE = 5
MIN_NEGATIVE = 5

def subgroup_metrics(df):
    n = len(df)
    positive = int((df["y_true"] == 1).sum())
    negative = int((df["y_true"] == 0).sum())

    result = {
        "N": n,
        "CKD": positive,
        "Non_CKD": negative,
        "Sensitivity": np.nan,
        "Specificity": np.nan,
        "PR_AUC": np.nan,
        "Status": "Insufficient cases"
    }

    if (positive < MIN_POSITIVE or negative < MIN_NEGATIVE):
        return pd.Series(result)

    y_true = df["y_true"].values
    y_pred = df["y_pred"].values
    y_prob = df["y_prob"].values

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    result["Sensitivity"] = (tp / (tp + fn))
    result["Specificity"] = (tn / (tn + fp))

    result["PR_AUC"] = average_precision_score(y_true, y_prob)
    result["Status"] = "Reported"

    return pd.Series(result)

# CKD Dataset results

In [14]:
ds1_results = []

for subgroup_type, column in [
    ("Gender", "gender_group"),
    ("Age group", "age_group")
]:

    for subgroup, group_df in ds1_predictions.groupby(column, dropna=False):

        metrics = subgroup_metrics(group_df)

        metrics["Dataset"] = "CKD"
        metrics["Subgroup_Type"] = subgroup_type
        metrics["Subgroup"] = subgroup

        ds1_results.append(metrics)

ds1_results = pd.DataFrame(ds1_results)

ds1_results = ds1_results[
    [
        "Dataset",
        "Subgroup_Type",
        "Subgroup",
        "N",
        "CKD",
        "Non_CKD",
        "Sensitivity",
        "Specificity",
        "PR_AUC",
        "Status"
    ]
]

print(ds1_results)

  Dataset Subgroup_Type Subgroup    N  CKD  Non_CKD  Sensitivity  Specificity  \
0     CKD        Gender   Female  195  133       62     0.932331     0.822581   
1     CKD        Gender     Male  185  106       79     0.971698     0.848101   
2     CKD     Age group    40-49   78   56       22     0.946429     0.727273   
3     CKD     Age group      50+   51   41       10     0.951220     0.500000   
4     CKD     Age group      <40  251  142      109     0.950704     0.889908   

     PR_AUC    Status  
0  0.984656  Reported  
1  0.979740  Reported  
2  0.974546  Reported  
3  0.974149  Reported  
4  0.987468  Reported  


In [15]:
ds1_results.to_csv(
    "exp6_CKD_subgroup_results.csv",
    index=False
)

# DiabeticCKD dataset results

In [16]:
ds2_results = []

for subgroup_type, column in [
    ("Gender", "gender_group"),
    ("Age group", "age_group")
]:

    for subgroup, group_df in ds2_predictions.groupby(column, dropna=False):

        metrics = subgroup_metrics(group_df)

        metrics["Dataset"] = "DiabeticCKD"
        metrics["Subgroup_Type"] = subgroup_type
        metrics["Subgroup"] = subgroup

        ds2_results.append(metrics)

ds2_results = pd.DataFrame(ds2_results)

ds2_results = ds2_results[
    [
        "Dataset",
        "Subgroup_Type",
        "Subgroup",
        "N",
        "CKD",
        "Non_CKD",
        "Sensitivity",
        "Specificity",
        "PR_AUC",
        "Status"
    ]
]

print(ds2_results)

       Dataset Subgroup_Type Subgroup     N  CKD  Non_CKD  Sensitivity  \
0  DiabeticCKD        Gender   Female  2550  247     2303     0.526316   
1  DiabeticCKD        Gender     Male  1450  139     1311     0.546763   
2  DiabeticCKD     Age group    40-49  1240  103     1137     0.485437   
3  DiabeticCKD     Age group      50+  2184  259     1925     0.571429   
4  DiabeticCKD     Age group      <40   576   24      552     0.333333   

   Specificity    PR_AUC    Status  
0     0.924012  0.455085  Reported  
1     0.909992  0.442689  Reported  
2     0.926121  0.440327  Reported  
3     0.899221  0.470462  Reported  
4     0.972826  0.313125  Reported  


In [17]:
ds2_results.to_csv(
    "exp6_DiabeticCKD_subgroup_results.csv",
    index=False
)